# Merge Real + Synthetic PSA Datasets (consistent IDs, Google Drive)

**Updated:** every row now gets ONE consistent sequential ID (`PSA_00001` ... `PSA_19816`) regardless of source - `Data_Source` (moved to the last column) is now the only place that tracks whether a row is real or synthetic. Each row's original ID is preserved in `psa_reference_metadata.csv`'s `Original_PSA_ID` column, so nothing is lost.

**Main file schema:** `PSA_ID, Domain, English, Kiswahili, Ekegusii, Data_Source`

**Verified before packaging:** tested against your real files - 19,816 rows, all IDs confirmed matching the consistent `PSA_NNNNN` pattern, join-back to the reference file confirmed one-to-one, original IDs (e.g. `REAL_1`, `HEA_00001`) confirmed traceable.

**Before running:** make sure `kenyan_psa_15000_final.csv` and `df_with_swahili.csv` are already uploaded into your Google Drive's `NLP_Translation` folder.

## Step 1: Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Step 2: Run the merge

In [3]:
import pandas as pd
import os

# ------------------------------------------------------------------
# CONFIG - reads from and saves to Google Drive's NLP_Translation folder
# ------------------------------------------------------------------
DRIVE_FOLDER = "/content/drive/MyDrive/NLP_Translation"

SYNTHETIC_FILE = f"{DRIVE_FOLDER}/kenyan_psa_15000_final.csv"
REAL_FILE = f"{DRIVE_FOLDER}/df_with_swahili.csv"

MAIN_OUTPUT = f"{DRIVE_FOLDER}/psa_main_eng_kiswahili_ekegusii.csv"
REFERENCE_OUTPUT = f"{DRIVE_FOLDER}/psa_reference_metadata.csv"

for required_file, label in [(SYNTHETIC_FILE, "synthetic file"), (REAL_FILE, "real file")]:
    if not os.path.exists(required_file):
        raise SystemExit(
            f"Could not find the {label} at: {required_file}\n"
            f"Upload it into your Google Drive's 'NLP_Translation' folder "
            f"(drive.google.com -> My Drive -> NLP_Translation -> upload), "
            f"or check the filename matches exactly (including capitalization)."
        )

# ------------------------------------------------------------------
# Load
# ------------------------------------------------------------------
final = pd.read_csv(SYNTHETIC_FILE, dtype=str)
ref = pd.read_csv(REAL_FILE, dtype=str)

print(f"Loaded synthetic file: {len(final)} rows")
print(f"Loaded real file: {len(ref)} rows")

required_synth_cols = ["PSA_ID", "Domain", "English", "Kiswahili", "Ekegusii",
                       "Kiswahili_tool", "Kiswahili_review_flag",
                       "Ekegusii_tool", "Ekegusii_review_flag",
                       "Dholuo", "Dholuo_tool", "Dholuo_review_flag", "Content_Type_Flag"]
missing_synth = [c for c in required_synth_cols if c not in final.columns]
if missing_synth:
    raise SystemExit(f"Missing column(s) in {SYNTHETIC_FILE}: {missing_synth}. "
                     f"Columns present: {list(final.columns)}")

required_real_cols = ["PSA_ID", "Domain", "en_clean", "sw_clean", "guz_clean",
                      "source_tag", "len_ratio", "en", "guz"]
missing_real = [c for c in required_real_cols if c not in ref.columns]
if missing_real:
    raise SystemExit(f"Missing column(s) in {REAL_FILE}: {missing_real}. "
                     f"Columns present: {list(ref.columns)}")

# ------------------------------------------------------------------
# Build the REAL rows (df_with_swahili.csv) - main + reference split.
# Original IDs are kept as plain "PSA_ID" values here temporarily (with
# a REAL_ prefix only to avoid collisions during the merge itself) -
# they get renamed to Original_PSA_ID further down, once the final
# consistent sequential ID is assigned.
# ------------------------------------------------------------------
real_main = pd.DataFrame({
    "PSA_ID": "REAL_" + ref["PSA_ID"].astype(str),
    "Domain": ref["Domain"],
    "Data_Source": "real_collected",
    "English": ref["en_clean"],
    "Kiswahili": ref["sw_clean"],
    "Ekegusii": ref["guz_clean"],
})
real_reference = pd.DataFrame({
    "PSA_ID": "REAL_" + ref["PSA_ID"].astype(str),
    "source_tag": ref["source_tag"],
    "len_ratio": ref["len_ratio"],
    "English_raw": ref["en"],
    "Ekegusii_raw": ref["guz"],
})

# ------------------------------------------------------------------
# Build the SYNTHETIC rows (kenyan_psa_15000_final.csv) - main + reference split
# ------------------------------------------------------------------
synth_main = pd.DataFrame({
    "PSA_ID": final["PSA_ID"],
    "Domain": final["Domain"],
    "Data_Source": "synthetic_generated",
    "English": final["English"],
    "Kiswahili": final["Kiswahili"],
    "Ekegusii": final["Ekegusii"],
})
synth_reference = pd.DataFrame({
    "PSA_ID": final["PSA_ID"],
    "Kiswahili_tool": final["Kiswahili_tool"],
    "Kiswahili_review_flag": final["Kiswahili_review_flag"],
    "Ekegusii_tool": final["Ekegusii_tool"],
    "Ekegusii_review_flag": final["Ekegusii_review_flag"],
    "Dholuo": final["Dholuo"],
    "Dholuo_tool": final["Dholuo_tool"],
    "Dholuo_review_flag": final["Dholuo_review_flag"],
    "Content_Type_Flag": final["Content_Type_Flag"],
})

# ------------------------------------------------------------------
# Combine, then assign ONE consistent sequential ID across every row.
# Data_Source (already present) is the reference column for tracking
# which source a row came from - the ID itself no longer encodes this.
# Each row's original ID is preserved in the reference file so nothing
# is lost, just moved out of the main working file.
# ------------------------------------------------------------------
main_df = pd.concat([real_main, synth_main], ignore_index=True)
reference_df = pd.concat([real_reference, synth_reference], ignore_index=True)

original_ids = main_df["PSA_ID"].tolist()  # capture before overwriting

n_rows = len(main_df)
id_width = len(str(n_rows))
new_ids = [f"PSA_{i+1:0{id_width}d}" for i in range(n_rows)]

main_df["PSA_ID"] = new_ids
reference_df["PSA_ID"] = new_ids
reference_df.insert(1, "Original_PSA_ID", original_ids)

# Data_Source moves to the LAST column - a reference/tracking field,
# not part of the core record identity.
main_df = main_df[["PSA_ID", "Domain", "English", "Kiswahili", "Ekegusii", "Data_Source"]]

assert main_df["PSA_ID"].duplicated().sum() == 0, "Unexpected duplicate PSA_IDs after merge!"
assert reference_df["PSA_ID"].duplicated().sum() == 0, "Unexpected duplicate PSA_IDs in reference file!"

main_df.to_csv(MAIN_OUTPUT, index=False)
reference_df.to_csv(REFERENCE_OUTPUT, index=False)

print(f"\nMain file saved: {MAIN_OUTPUT} ({len(main_df)} rows)")
print(f"Sample new IDs: {new_ids[0]} ... {new_ids[-1]}")
print(main_df["Data_Source"].value_counts())
print(f"\nReference file saved: {REFERENCE_OUTPUT} ({len(reference_df)} rows)")
print("Original per-source IDs preserved in reference file's 'Original_PSA_ID' column.")

# ------------------------------------------------------------------
# Coverage summary
# ------------------------------------------------------------------
def is_empty(s):
    return s.isna() | (s.astype(str).str.strip() == "")

print("\nCoverage by source:")
for source in ["real_collected", "synthetic_generated"]:
    sub = main_df[main_df["Data_Source"] == source]
    print(f"--- {source} ({len(sub)} rows) ---")
    for lang in ["English", "Kiswahili", "Ekegusii"]:
        filled = (~is_empty(sub[lang])).sum()
        print(f"  {lang}: {filled}/{len(sub)} ({filled/len(sub)*100:.1f}%)")


Loaded synthetic file: 15000 rows
Loaded real file: 4816 rows

Main file saved: /content/drive/MyDrive/NLP_Translation/psa_main_eng_kiswahili_ekegusii.csv (19816 rows)
Sample new IDs: PSA_00001 ... PSA_19816
Data_Source
synthetic_generated    15000
real_collected          4816
Name: count, dtype: int64

Reference file saved: /content/drive/MyDrive/NLP_Translation/psa_reference_metadata.csv (19816 rows)
Original per-source IDs preserved in reference file's 'Original_PSA_ID' column.

Coverage by source:
--- real_collected (4816 rows) ---
  English: 4816/4816 (100.0%)
  Kiswahili: 4816/4816 (100.0%)
  Ekegusii: 4816/4816 (100.0%)
--- synthetic_generated (15000 rows) ---
  English: 15000/15000 (100.0%)
  Kiswahili: 15000/15000 (100.0%)
  Ekegusii: 15000/15000 (100.0%)


## Step 3: Preview the results

In [4]:
import pandas as pd
main_df = pd.read_csv(MAIN_OUTPUT)
ref_df = pd.read_csv(REFERENCE_OUTPUT)
print('Main file (note Data_Source is the last column):')
print(main_df.head(3).to_string())
print()
print('Tracing PSA_00001 back to its original ID via the reference file:')
print(ref_df[ref_df['PSA_ID']=='PSA_00001'][['PSA_ID','Original_PSA_ID']].to_string())


Main file (note Data_Source is the last column):
      PSA_ID     Domain                                                                                                                                                                                             English                                                                                                                                                                                                                        Kiswahili                                                                                                                                                                                                                                                                                              Ekegusii     Data_Source
0  PSA_00001  Education  Introduces a multimodal AI pipeline that processes spoken English through speech-to-text, converts it into American Sign Language (ASL) gloss, and generates lifelike 3D si

## Done

Both output files are saved directly in your Drive's `NLP_Translation` folder.